<a href="https://colab.research.google.com/github/yeseul-04/kfa-ai-agent/blob/main/w04_%EA%B2%80%EC%A6%9D%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔎 4회차 검증기: AI가 쓴 숫자를 원본과 대조하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phm0305/KFA-ai-agent-study/blob/main/notebooks/04_checker.ipynb)

글에 나온 숫자가 **어느 원본 줄에서 왔는지** Gemini가 고르고, 맞는지는 **코드가** 원본과 비교해서 판정해요.

```
DART 2개 연도 ──▶ rid · [S#] ──▶ 지표 [M#] ──▶ Gemini가 주장·근거 뽑기 ──▶ 답 원문 저장
                                                                      │
                     결과 파일 ◀── 대조 (PASS / FIX / REVIEW) ◀── 저장본 읽기 ◀┘
```

- **PASS** 숫자가 원본과 맞음 · **FIX** 틀림(실제 값을 옆에 보여줘요) · **REVIEW** 사람이 봐야 함(근거 없음, 단위 문제 등)

**사용법:** 아래 설정 셀의 `STOCK_CODE`, `YEAR`, `TEXT` 세 개만 바꾸고 위 메뉴 **런타임 → 모두 실행**.
"보안 비밀 액세스 허용" 창이 뜨면 **허용**을 눌러요. 중간에 뭐가 실패해도 멈추지 않고 끝까지 가요.

**설정**

- `STOCK_CODE`: 종목코드 6자리. 따옴표 안에, 앞의 0까지 적어요
- `YEAR`: 사업보고서 연도. 이 해와 그 전년도 두 해를 받아요
- `TEXT`: 검증할 글. 기본값은 SK하이닉스 2025년으로 미리 시험해 본 다섯 문장이에요(일부러 틀린 숫자 52%가 들어 있어요). 내 글로 바꿔 넣어도 돼요

In [1]:
STOCK_CODE = "000660"
YEAR = 2025
TEXT = """
SK하이닉스의 2025년 매출은 97.1조 원으로 전년보다 46.8% 늘었다. 영업이익률은 52%를 기록했고, 전년 대비 13.1%포인트 높아졌다. 부채비율은 46% 수준이다. 부채비율은 전년보다 16.2%p 낮아졌다. HBM 수요가 실적을 이끌었다.
"""

In [2]:
#@title 📦 설치 (1분) { display-mode: "form" }
import subprocess, sys
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "finance-datareader", "python-dotenv", "google-genai"],
                      capture_output=True, text=True)
print("✅ 설치 완료" if proc.returncode == 0 else "❌ 설치 실패: 런타임 → 런타임 다시 시작 → 모두 실행")

✅ 설치 완료


In [3]:
#@title 🧰 공통 도구 (고치지 말고 실행만. 코드가 궁금하면 이 셀을 더블클릭) { display-mode: "form" }
"""KFA AI agent study: shared collection toolkit (2nd session template).

Every source, whatever its collection level (1 API, 2 library, 3 file,
4 crawling, 5 manual), ends in the same record table via finalize().
Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 목차] 이 셀은 고치지 않고 실행만 해요. 아래는 어떤 함수가 어디에 쓰이는지예요.

  공통 출구 (모든 단계가 마지막에 거치는 곳)
    collect_table()     받아온 표 → 공통 표 15칸. 내 소스 함수의 마지막 줄은 항상 이것
    finalize()          꼬리표 붙이기·숫자/날짜 정리·형식 통과/확인 필요 판정 (collect_table이 불러 씀)
    long_from_table()   칸마다 항목이 하나씩인 넓은 표 → 숫자 하나 = 한 줄 (collect_table이 불러 씀)
    check_records()     공통 표 모양이 맞는지 검사하고 요약 출력
    save_records()      CSV 두 개로 저장 (전부 / 확인 필요만)
    run_step()          예시 하나를 실행하고 ✅ ⏭ ❌ 요약표에 한 줄 남김 (실패해도 멈추지 않음)
    step_summary()      요약표 보여주기

  1단계 공식 API      read_json_table() · collect_dart() · collect_ecos() · get_key()
  2단계 라이브러리     (도구 없음: 라이브러리가 준 표를 collect_table에 바로)
  3단계 파일          read_table()
  4단계 크롤링        read_html_table() · robots_allowed() · find_col()
  5단계 수동 입력      (pd.DataFrame으로 직접) · mount_drive() · read_source_cards() · pick_sheet_title() · collect_from_cards()

  정리 도구           to_number() 숫자 바꾸기 · to_date() 날짜 바꾸기 · is_blank() 빈칸 판단
  약속된 값           NOT_APPLICABLE = "해당 없음" · SAME_AS_DATE = 공시일을 기준일과 같게
  멈춤 신호           SkipStep = 준비 부족이라 건너뜀(⏭) · CollectError = 실패(❌), 이유를 쉬운 말로
"""

import io
import json
import math
import numbers
import os
import re
import urllib.robotparser
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

COLUMNS = [
    "종목코드", "종목명", "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각", "상태", "비고",
]
REQUIRED = [
    "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각",
]
NOT_APPLICABLE = "해당 없음"
NA_ALLOWED = {"종목코드", "종목명", "기준", "공시일"}
RELIABILITY = ("상", "중", "하")
LEVELS = {1: "공식 API", 2: "파이썬 라이브러리", 3: "파일 다운로드", 4: "크롤링", 5: "수동 입력"}
PASSED, REVIEW = "형식 통과", "확인 필요"
SAME_AS_DATE = "same_as_date"
CHECKED_MARKS = {"o", "y", "yes", "✅", "확인"}
NULL_TOKENS = {"", "-", ".", "\u2014", "n/a", "na", "nan", "none", "null"}
KST = timezone(timedelta(hours=9))
USER_AGENT = "KFA-AI-agent-study/0.2 (+https://github.com/phm0305/KFA-ai-agent-study)"


# 실패(❌). 메시지가 조원 화면에 그대로 나가서 키·전체 주소를 넣지 않아요
class CollectError(Exception):
    """Shown to members verbatim, so it must never contain keys or full URLs."""


# 준비가 안 돼서 건너뜀(⏭). 키 없음, 설정 꺼짐, 사이트가 막음 등
class SkipStep(Exception):
    pass


# 지금 한국 시각 (수집 시각 칸에 씀)
def now_kst():
    return datetime.now(KST).strftime("%Y-%m-%d %H:%M")


# 빈칸인지 판단 (None, 빈 글자, NaN 모두 빈칸)
def is_blank(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


# '1,234' '(500)' '△500' '12.5%' → 숫자. 못 바꾸면 (None, 이유)
def to_number(value):
    if is_blank(value):
        return None, None
    if isinstance(value, bool):
        return None, f"숫자로 못 바꿈: '{value}'"
    if isinstance(value, numbers.Integral):
        return int(value), None
    if isinstance(value, numbers.Real):
        return (float(value), None) if math.isfinite(value) else (None, None)
    text = str(value).strip()
    if text.lower() in NULL_TOKENS:
        return None, None
    cleaned = text.replace(",", "").replace(" ", "").rstrip("%")
    negative = False
    if cleaned.startswith("(") and cleaned.endswith(")"):
        negative, cleaned = True, cleaned[1:-1]
    elif cleaned.startswith("△"):
        negative, cleaned = True, cleaned[1:]
    if re.fullmatch(r"[+-]?\d+", cleaned):
        number = int(cleaned)
    else:
        try:
            number = float(cleaned)
        except ValueError:
            return None, f"숫자로 못 바꿈: '{text}'"
        if not math.isfinite(number):
            return None, f"숫자로 못 바꿈: '{text}'"
    return (-number if negative else number), None


_DAY = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*[-./월]\s*(\d{1,2})\s*일?\.?")
_COMPACT_DAY = re.compile(r"(\d{4})(\d{2})(\d{2})")
_MONTH = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*월?\.?")
_COMPACT_MONTH = re.compile(r"(\d{4})(\d{2})")
_QUARTER = re.compile(r"(\d{4})\s*-?\s*Q([1-4])", re.IGNORECASE)
_YEAR = re.compile(r"(\d{4})\s*년?")
_TIME_SUFFIX = re.compile(r"[ T]\d{1,2}:\d{2}(:\d{2})?(\.\d+)?$")


# '2026.6.30.' '20260630' '2026년 6월' '2026Q2' 등 → 표준 날짜 글자. 못 바꾸면 (None, 이유)
def to_date(value):
    """Normalize to YYYY-MM-DD, YYYY-MM, YYYYQn or YYYY; anything else is a problem."""
    if is_blank(value):
        return None, None
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.strftime("%Y-%m-%d"), None
    text = _TIME_SUFFIX.sub("", str(value).strip())
    for pattern in (_DAY, _COMPACT_DAY):
        match = pattern.fullmatch(text)
        if match:
            try:
                return date(*map(int, match.groups())).isoformat(), None
            except ValueError:
                return None, f"없는 날짜: '{value}'"
    for pattern in (_MONTH, _COMPACT_MONTH):
        match = pattern.fullmatch(text)
        if match:
            year, month = map(int, match.groups())
            if 1 <= month <= 12:
                return f"{year:04d}-{month:02d}", None
            return None, f"없는 날짜: '{value}'"
    match = _QUARTER.fullmatch(text)
    if match:
        return f"{match.group(1)}Q{match.group(2)}", None
    match = _YEAR.fullmatch(text)
    if match:
        return match.group(1), None
    return None, f"날짜 모양을 모름: '{value}'"


# 이름에 그 글자가 들어간 첫 칸 찾기 (크롤링 표처럼 칸 이름이 조금씩 바뀔 때)
def find_col(table, keyword):
    for col in table.columns:
        if keyword in str(col):
            return col
    raise CollectError(f"'{keyword}'가 들어간 칸이 없어요. 지금 표의 칸: {[str(c) for c in table.columns]}")


# 넓은 표(칸마다 항목) → 긴 표(숫자 하나 = 한 줄). items·date_col을 여기서 씀
def long_from_table(table, items, date_col=None, keep=None):
    """Wide table (one column per item) -> one row per number.

    items: {raw column: (항목 이름, 단위)}. keep: {raw column: our column} copied per row.
    """
    if date_col is None:
        if isinstance(table.index, pd.RangeIndex):
            raise CollectError("날짜가 들어 있는 칸 이름을 date_col로 알려 주세요")
        table = table.reset_index()
        date_col = table.columns[0]
    keep = keep or {}
    missing = [c for c in [*items, date_col, *keep] if c not in table.columns]
    if missing:
        raise CollectError(f"표에 없는 칸: {missing}. 지금 표의 칸: {[str(c) for c in table.columns]}")
    parts = []
    for raw_col, (label, unit) in items.items():
        part = pd.DataFrame({
            "항목": label,
            "값": pd.Series(table[raw_col].tolist(), dtype=object),
            "단위": unit,
            "기준일": pd.Series(table[date_col].tolist(), dtype=object),
        })
        for raw, ours in keep.items():
            part[ours] = table[raw].tolist()
        parts.append(part)
    if not parts:
        raise CollectError("items가 비어 있어요. 어떤 칸을 숫자로 가져올지 적어 주세요")
    return pd.concat(parts, ignore_index=True)


# 꼬리표 채우기 + 숫자·날짜 정리 + 형식 통과/확인 필요 판정 (모든 단계 공통)
def finalize(table, *, level, source=None, source_url=None, reliability=None, basis=None,
             disclosed=None, base_date=None, stock_code=None, stock_name=None, unit=None, note=None):
    """Turn any table with at least 항목/값 into the shared record table.

    A cell already in the table wins over the keyword default. disclosed may be a
    date, NOT_APPLICABLE, or SAME_AS_DATE (copy 기준일, for prices and rates).
    """
    if level not in LEVELS:
        raise ValueError(f"level must be one of {list(LEVELS)}")
    defaults = {
        "종목코드": stock_code, "종목명": stock_name, "단위": unit, "기준": basis,
        "기준일": base_date, "출처": source, "출처 URL": source_url, "신뢰도": reliability,
    }
    collected_at = now_kst()
    method = f"{level} {LEVELS[level]}"
    out = []
    for raw in table.to_dict("records"):
        row = {col: raw.get(col) for col in COLUMNS}
        for col, default in defaults.items():
            if is_blank(row[col]) and default is not None:
                row[col] = default
        notes = [n for n in (raw.get("비고"), note) if not is_blank(n)]
        reasons = []

        if is_blank(row["공시일"]):
            if disclosed == SAME_AS_DATE:
                row["공시일"] = row["기준일"]
                notes.append("공시일은 기준일과 같다고 봄")
            elif disclosed is not None:
                row["공시일"] = disclosed
        row["수집 방법"] = method
        if is_blank(row["수집 시각"]):
            row["수집 시각"] = collected_at

        value, value_problem = to_number(row["값"])
        if value_problem:
            reasons.append(value_problem)
        else:
            row["값"] = value

        for col in ("기준일", "공시일"):
            if is_blank(row[col]) or row[col] == NOT_APPLICABLE:
                continue
            normalized, problem = to_date(row[col])
            if problem:
                reasons.append(f"{col} {problem}")
            else:
                row[col] = normalized

        for col in REQUIRED:
            if is_blank(row[col]) and not (col == "값" and value_problem):
                reasons.append(f"{col} 비어 있음")
        for col in COLUMNS:
            if row[col] == NOT_APPLICABLE and col not in NA_ALLOWED:
                reasons.append(f"{col}에는 '{NOT_APPLICABLE}'을 쓸 수 없음")

        base, disc = str(row["기준일"] or ""), str(row["공시일"] or "")
        if re.fullmatch(r"\d{4}-\d{2}(-\d{2})?", base) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", disc):
            if disc[:len(base)] < base:
                reasons.append("공시일이 기준일보다 빠름")
        if not is_blank(row["신뢰도"]) and row["신뢰도"] not in RELIABILITY:
            reasons.append("신뢰도는 상/중/하 중 하나")

        checked = str(raw.get("확인") or "").strip().lower() in CHECKED_MARKS
        if level == 4 and not checked:
            reasons.append("크롤링 값: 원문 화면과 한 번 대조")
        if level == 5 and not checked:
            reasons.append("손으로 옮긴 값: 원문과 대조")

        row["상태"] = REVIEW if reasons else PASSED
        row["비고"] = "; ".join([*dict.fromkeys(notes), *(f"⚠ {r}" for r in reasons)])
        out.append(row)

    records = pd.DataFrame(out, columns=COLUMNS)
    records["값"] = pd.Series([r["값"] for r in out], dtype=object)
    return records


# 모든 단계의 공통 출구: 받아온 표 → 공통 표 15칸
def collect_table(table, items=None, date_col=None, keep=None, **meta):
    """Single exit for every level: raw table in, record table out.

    Without items, the table must already have 항목/값 columns (typical for manual CSVs).
    """
    if not isinstance(table, pd.DataFrame):
        raise CollectError(f"표(DataFrame)가 아니라 {type(table).__name__}가 들어왔어요")
    if items is not None:
        table = long_from_table(table, items, date_col, keep)
    elif not {"항목", "값"} <= set(table.columns):
        raise CollectError(f"items 없이 쓰려면 '항목'과 '값' 칸이 있어야 해요. 지금 칸: {[str(c) for c in table.columns]}")
    return finalize(table, **meta)


# 확인 필요 줄의 ⚠ 이유를 종류별로 세기
def review_reasons(records):
    reasons = Counter()
    for note in records.loc[records["상태"] == REVIEW, "비고"].fillna(""):
        for part in note.split("; "):
            if part.startswith("⚠ "):
                reasons[re.sub(r"'.*'", "'…'", part[2:])] += 1
    return reasons


# 공통 표 모양 검사. 맞으면 '✅ 칸 모양 OK' 요약을 출력
def check_records(records, show=True):
    problems = []
    if not isinstance(records, pd.DataFrame):
        problems.append(f"표(DataFrame)가 아니에요: {type(records).__name__}")
    elif list(records.columns) != COLUMNS:
        missing = [c for c in COLUMNS if c not in records.columns]
        extra = [c for c in records.columns if c not in COLUMNS]
        problems.append(f"칸 이름·순서가 달라요 (빠진 칸 {missing}, 남는 칸 {extra})")
    else:
        bad_state = ~records["상태"].isin([PASSED, REVIEW])
        if bad_state.any():
            problems.append(f"상태 칸에 '{PASSED}'/'{REVIEW}' 말고 다른 값이 {int(bad_state.sum())}줄")
        blank = records[REQUIRED].apply(lambda col: col.map(is_blank)).any(axis=1)
        hidden = blank & (records["상태"] == PASSED)
        if hidden.any():
            problems.append(f"필수 칸이 비었는데 '{PASSED}'인 줄이 {int(hidden.sum())}개 (finalize를 거치지 않은 듯)")
    if show:
        if problems:
            for p in problems:
                print("❌", p)
        else:
            passed = int((records["상태"] == PASSED).sum())
            print(f"✅ 칸 모양 OK: {len(records)}줄 (형식 통과 {passed} · 확인 필요 {len(records) - passed})")
            for reason, count in review_reasons(records).most_common(5):
                print(f"   ⚠ {reason}: {count}줄")
    return not problems


# CSV 저장: 전부(stem.csv) + 확인 필요만(stem_확인필요.csv). 엑셀에서 안 깨지게 utf-8-sig
def save_records(records, stem="records"):
    all_path, review_path = f"{stem}.csv", f"{stem}_확인필요.csv"
    records.to_csv(all_path, index=False, encoding="utf-8-sig")
    records[records["상태"] == REVIEW].to_csv(review_path, index=False, encoding="utf-8-sig")
    return all_path, review_path


STEP_RESULTS = []


# 예시 하나 실행 → 요약표에 ✅ ⏭ ❌ 한 줄. 어떤 오류가 나도 '모두 실행'을 멈추지 않음
def run_step(label, level, func):
    """Run one source without ever stopping 'Run all'; errors become a summary row."""
    entry = {"단계": f"{level} {LEVELS[level]}", "예시": label, "결과": "❌", "줄 수": 0, "확인 필요": 0, "메모": ""}
    STEP_RESULTS[:] = [e for e in STEP_RESULTS if e["예시"] != label]
    STEP_RESULTS.append(entry)
    try:
        records = func()
    except SkipStep as e:
        entry.update(결과="⏭", 메모=str(e))
        print(f"⏭ {label}: {e}")
        return None
    except CollectError as e:
        entry["메모"] = str(e)
        print(f"❌ {label}: {e}")
        return None
    except requests.RequestException as e:
        entry["메모"] = f"인터넷 연결 문제로 못 받아왔어요 ({type(e).__name__}). 잠시 뒤 이 셀만 다시 실행해 주세요"
        print(f"❌ {label}: {entry['메모']}")
        return None
    except Exception as e:
        entry["메모"] = f"예상 못 한 오류 ({type(e).__name__}). 셀을 다시 실행해 보고, 그래도 안 되면 조 톡방에 공유"
        print(f"❌ {label}: {entry['메모']}")
        return None
    if not check_records(records, show=False):
        entry["메모"] = "공통 표 모양이 아니에요. check_records(결과)로 무엇이 다른지 확인"
        check_records(records)
        return None
    review = int((records["상태"] == REVIEW).sum())
    entry.update(결과="✅", **{"줄 수": len(records), "확인 필요": review})
    print(f"✅ {label}: {len(records)}줄 (확인 필요 {review}줄)")
    return records


# 지금까지 실행한 예시의 요약표
def step_summary():
    return pd.DataFrame(STEP_RESULTS, columns=["단계", "예시", "결과", "줄 수", "확인 필요", "메모"])


# 주소에 요청 보내기. 오류 메시지에는 사이트 이름만 넣어 키가 화면에 안 나오게 함
def http_get(url, params=None, timeout=30):
    host = urlparse(url).netloc
    try:
        resp = requests.get(url, params=params, timeout=timeout, headers={"User-Agent": USER_AGENT})
    except requests.Timeout:
        raise CollectError(f"{host} 응답이 {timeout}초 넘게 없어요. 잠시 뒤 다시 실행해 주세요") from None
    except requests.RequestException:
        raise CollectError(f"{host}에 연결하지 못했어요. 인터넷이나 사이트 상태를 확인해 주세요") from None
    if resp.status_code != 200:
        hint = {401: "키나 로그인이 필요해요", 403: "사이트가 접근을 막았어요", 404: "주소가 바뀌었을 수 있어요",
                429: "요청이 너무 잦아요. 1분 뒤 다시"}.get(resp.status_code, "")
        raise CollectError(f"{host}가 {resp.status_code} 응답을 줬어요. {hint}".strip())
    return resp


# [1단계] JSON 응답에서 'a.b.c' 위치의 목록을 표로. 위치가 틀리면 그 자리에 뭐가 있는지 알려줌
def read_json_table(url, rows_path, params=None):
    """Fetch JSON and return the list found at a dotted path (e.g. 'StatisticSearch.row') as a table."""
    try:
        data = http_get(url, params).json()
    except ValueError:
        raise CollectError("응답이 JSON이 아니에요. 주소의 형식(json/xml) 부분을 확인해 주세요") from None
    current = data
    for key in rows_path.split("."):
        if isinstance(current, dict) and key in current:
            current = current[key]
            continue
        here = list(current) if isinstance(current, dict) else type(current).__name__
        preview = json.dumps(data, ensure_ascii=False)[:200]
        raise CollectError(f"응답에서 '{key}'를 못 찾았어요. 그 자리에 있는 것: {here} / 응답 앞부분: {preview}")
    rows = [current] if isinstance(current, dict) else current
    if not isinstance(rows, list) or not rows:
        raise CollectError(f"'{rows_path}' 아래가 비어 있어요")
    return pd.json_normalize(rows)


DART_URL = "https://opendart.fss.or.kr/api"
DART_REPORT = {1: "11013", 2: "11012", 3: "11014", 4: "11011"}
QUARTER_END = {1: "03-31", 2: "06-30", 3: "09-30", 4: "12-31"}
DART_ERRORS = {
    "010": "등록되지 않은 키예요. Secrets의 키 값을 확인해 주세요",
    "011": "사용할 수 없는 키예요",
    "013": "조회된 데이터가 없어요. 아직 공시 전인 분기인지 확인해 주세요",
    "020": "오늘 요청 한도를 넘었어요. 내일 다시 해 주세요",
    "100": "요청 값이 잘못됐어요 (연도·분기·연결/별도)",
    "800": "DART 시스템 점검 중이에요",
}
DART_ITEMS = {
    "ifrs-full_Revenue": ("매출액", True),
    "dart_OperatingIncomeLoss": ("영업이익", True),
    "ifrs-full_ProfitLoss": ("당기순이익", True),
    "ifrs-full_Assets": ("자산총계", False),
    "ifrs-full_Liabilities": ("부채총계", False),
    "ifrs-full_Equity": ("자본총계", False),
}
REPO_RAW_URL = "https://raw.githubusercontent.com/phm0305/KFA-ai-agent-study/main/"
DART_CORPS_FILE = "backup/dart_corps.csv"
_DART_CORPS = {}
_DART_CORPS_LOADED = set()


def _dart_status(data):
    status = data.get("status")
    if status != "000":
        raise CollectError(f"DART {status}: {DART_ERRORS.get(status, data.get('message', '알 수 없는 응답'))}")


def download_dart_corps(api_key):
    """Listed companies from DART's full corpCode.xml: {stock_code: (corp_code, corp_name)}."""
    resp = http_get(f"{DART_URL}/corpCode.xml", {"crtfc_key": api_key}, timeout=600)
    try:
        xml_bytes = zipfile.ZipFile(io.BytesIO(resp.content)).read("CORPCODE.xml")
    except zipfile.BadZipFile:
        try:
            _dart_status(resp.json())
        except ValueError:
            pass
        raise CollectError("DART 회사 번호표를 받지 못했어요. 키를 확인해 주세요") from None
    corps = {}
    for item in ET.fromstring(xml_bytes).iter("list"):
        code = (item.findtext("stock_code") or "").strip()
        if code:
            corps[code] = (item.findtext("corp_code"), item.findtext("corp_name"))
    return corps


def read_dart_corps(content):
    # dtype=str keeps leading zeros: read as numbers, "000660" would become 660 and never match.
    table = pd.read_csv(io.BytesIO(content), dtype=str, keep_default_na=False)
    return {row.stock_code: (row.corp_code, row.corp_name) for row in table.itertuples(index=False)}


# [1단계] 종목코드 → DART 회사 번호. 레포의 작은 상장사 표를 먼저 보고, 없을 때만 DART 전체 목록을 받음
def dart_corp(stock_code, api_key):
    if "repo" not in _DART_CORPS_LOADED:
        _DART_CORPS_LOADED.add("repo")
        try:
            _DART_CORPS.update(read_dart_corps(http_get(REPO_RAW_URL + DART_CORPS_FILE, timeout=10).content))
        except CollectError:
            pass
    if stock_code not in _DART_CORPS and "dart" not in _DART_CORPS_LOADED:
        print("DART 회사 목록을 받는 중이에요. 처음 한 번, 몇 분 걸릴 수 있어요")
        _DART_CORPS.update(download_dart_corps(api_key))
        _DART_CORPS_LOADED.add("dart")
    if stock_code not in _DART_CORPS:
        raise CollectError(f"{stock_code}: 상장사 목록에 없어요. 종목코드 6자리를 확인해 주세요")
    return _DART_CORPS[stock_code]


# [1단계] DART 재무제표 응답에서 6개 계정만 골라 3개월·누적을 나눠 적기
def dart_rows_to_long(rows, year, quarter):
    """Pick six accounts by standard account_id; flows get 3-month and cumulative rows."""
    base_note = "기준일은 12월 결산 가정"
    out, seen = [], set()
    for row in rows:
        account = row.get("account_id")
        if account not in DART_ITEMS or account in seen:
            continue
        name, is_flow = DART_ITEMS[account]
        if is_flow and row.get("sj_div") not in ("IS", "CIS"):
            continue
        seen.add(account)
        rcept_no = row.get("rcept_no", "")
        common = {
            "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
            "공시일": rcept_no[:8], "출처 URL": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
        }
        if not is_flow:
            parts = [(name, row.get("thstrm_amount"), base_note)]
        elif quarter == 4:
            parts = [(f"{name}(연간)", row.get("thstrm_amount"), base_note + "; 4분기 3개월치 = 연간 - 3분기 누적")]
        elif quarter == 1:
            parts = [(f"{name}(3개월)", row.get("thstrm_amount"), base_note + "; 1분기는 3개월 = 누적")]
        else:
            parts = [(f"{name}(3개월)", row.get("thstrm_amount"), base_note),
                     (f"{name}(누적)", row.get("thstrm_add_amount"), base_note)]
        for label, amount, memo in parts:
            out.append({**common, "항목": label, "값": amount, "비고": memo})
    for account, (name, _) in DART_ITEMS.items():
        if account not in seen:
            out.append({"항목": name, "값": None, "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
                        "비고": f"이 회사 공시에서 표준 계정({account})을 못 찾음"})
    return pd.DataFrame(out)


# [1단계] DART 재무제표 → 공통 표 (매출액·영업이익·당기순이익·자산·부채·자본)
def collect_dart(stock_code, year, quarter, fs_div, api_key):
    if not api_key:
        raise SkipStep("DART 키가 없어요 (Secrets에 DART_API_KEY 등록 후 다시)")
    if quarter not in DART_REPORT or fs_div not in ("CFS", "OFS"):
        raise CollectError("QUARTER는 1~4, FS_DIV는 'CFS'(연결) 또는 'OFS'(별도)")
    corp_code, corp_name = dart_corp(stock_code, api_key)
    data = http_get(f"{DART_URL}/fnlttSinglAcntAll.json", {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": str(year),
        "reprt_code": DART_REPORT[quarter], "fs_div": fs_div,
    }).json()
    _dart_status(data)
    return collect_table(
        dart_rows_to_long(data["list"], year, quarter),
        level=1, source="DART 단일회사 전체 재무제표 API", reliability="상",
        basis="연결" if fs_div == "CFS" else "별도", stock_code=stock_code, stock_name=corp_name,
        source_url="https://opendart.fss.or.kr",
    )


# [1단계] 한국은행 ECOS 통계 1개 → 공통 표. 키가 없으면 ⏭
def collect_ecos(api_key, stat_code, cycle, start, end, item_code, label, unit):
    if not api_key:
        raise SkipStep("ECOS 키가 없어요 (선택 사항: ecos.bok.or.kr에서 무료 발급 후 Secrets에 ECOS_API_KEY)")
    url = f"https://ecos.bok.or.kr/api/StatisticSearch/{api_key}/json/kr/1/1000/{stat_code}/{cycle}/{start}/{end}/{item_code}"
    try:
        data = http_get(url).json()
    except ValueError:
        raise CollectError("ECOS 응답이 JSON이 아니에요") from None
    if "RESULT" in data:
        raise CollectError(f"ECOS: {data['RESULT'].get('MESSAGE', data['RESULT'])}")
    table = pd.json_normalize(data["StatisticSearch"]["row"])
    return collect_table(
        table, {"DATA_VALUE": (label, unit)}, date_col="TIME",
        level=1, source="한국은행 ECOS Open API", reliability="상", basis=NOT_APPLICABLE,
        source_url=f"https://ecos.bok.or.kr/#/SearchStat?statCode={stat_code}",
    )


# [3·5단계] 주소나 파일 경로의 CSV·엑셀 읽기 (한글 깨짐 cp949, 이름만 .xls인 파일도 처리)
# 칸 이름과 글자 값의 앞뒤 공백은 지워요 (화면에서 안 보이는 차이라서)
def read_table(source, sheet_name=0, skiprows=None):
    """CSV/Excel from a URL or a path. Handles cp949 CSVs and HTML disguised as .xls.

    Leading/trailing spaces in column names and text cells are removed: they are
    invisible on screen, so no reader can mean anything by them.
    """
    table = _read_table_raw(source, sheet_name, skiprows)
    table.columns = [c.strip() if isinstance(c, str) else c for c in table.columns]
    for col in table.columns:
        if not pd.api.types.is_numeric_dtype(table[col]):
            table[col] = table[col].map(lambda v: v.strip() if isinstance(v, str) else v)
    return table


def _read_table_raw(source, sheet_name, skiprows):
    if str(source).startswith(("http://", "https://")):
        content, name = http_get(str(source)).content, urlparse(str(source)).path.lower()
    else:
        path = Path(source)
        if not path.exists():
            raise CollectError(f"파일이 없어요: {path.name}. 경로와 파일 이름(띄어쓰기·확장자)을 확인해 주세요")
        content, name = path.read_bytes(), path.name.lower()
    if content.lstrip()[:1] == b"<":
        try:
            return pd.read_html(io.StringIO(content.decode("utf-8", errors="replace")))[0]
        except ValueError:
            raise CollectError("파일 안이 표가 아닌 웹페이지예요. 받은 파일을 열어 확인해 주세요") from None
    if name.endswith((".xlsx", ".xlsm", ".xls")):
        try:
            return pd.read_excel(io.BytesIO(content), sheet_name=sheet_name, skiprows=skiprows)
        except ImportError:
            raise CollectError("이 엑셀 형식을 읽는 도구가 없어요. 엑셀에서 .xlsx로 다시 저장해 주세요") from None
    for encoding in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(io.BytesIO(content), encoding=encoding, skiprows=skiprows)
        except UnicodeDecodeError:
            continue
    raise CollectError("CSV 글자를 못 읽었어요 (utf-8, cp949 둘 다 실패). 엑셀에서 'CSV UTF-8'로 다시 저장해 주세요")


# [4단계] 사이트의 robots.txt가 코드 수집을 허락하는지 (모르면 None)
def robots_allowed(url):
    """True/False from robots.txt; None when it cannot be read (caller decides)."""
    parts = urlparse(url)
    try:
        resp = requests.get(f"{parts.scheme}://{parts.netloc}/robots.txt", timeout=15,
                            headers={"User-Agent": USER_AGENT})
    except requests.RequestException:
        return None
    if resp.status_code in (401, 403):
        return False
    if resp.status_code >= 400:
        return True
    parser = urllib.robotparser.RobotFileParser()
    parser.parse(resp.text.splitlines())
    return parser.can_fetch(USER_AGENT, url)


# [4단계] 웹페이지 화면의 표 읽기. robots.txt가 막으면 ⏭
def read_html_table(url, match=".+", index=0):
    host = urlparse(url).netloc
    allowed = robots_allowed(url)
    if allowed is False:
        raise SkipStep(f"{host}의 robots.txt가 코드 수집을 막고 있어요. 이 자료는 5단계(수동)로 옮겨 주세요")
    if allowed is None:
        print(f"⚠ {host}의 robots.txt를 확인하지 못했어요. 사이트 이용약관을 직접 확인해 주세요")
    resp = http_get(url)
    if resp.encoding is None or resp.encoding.lower() == "iso-8859-1":
        resp.encoding = resp.apparent_encoding
    try:
        tables = pd.read_html(io.StringIO(resp.text), match=match)
    except ValueError:
        raise CollectError(f"페이지에서 '{match}'가 들어간 표를 못 찾았어요. 표가 자바스크립트로 그려지는 페이지일 수 있어요") from None
    if index >= len(tables):
        raise CollectError(f"표가 {len(tables)}개뿐이에요. index를 0~{len(tables) - 1} 중에서 골라 주세요")
    table = tables[index]
    if isinstance(table.columns, pd.MultiIndex):
        table.columns = [" ".join(dict.fromkeys(str(c) for c in col)) for col in table.columns]
    return table


DRIVE_ROOT = "/content/drive/MyDrive/KFA_자료"
CARD_COLUMNS = ["파일명", "출처", "URL", "기준일", "신뢰도"]


# [5단계] 구글 드라이브 연결 후 KFA_자료 폴더 위치 돌려주기 (Colab 전용)
def mount_drive():
    try:
        from google.colab import drive
    except ImportError:
        raise CollectError("구글 드라이브 연결은 Colab에서만 돼요") from None
    drive.mount("/content/drive")
    if not Path(DRIVE_ROOT).exists():
        raise CollectError("내 드라이브 맨 위에 'KFA_자료' 폴더가 없어요 (이름의 밑줄까지 똑같이)")
    return Path(DRIVE_ROOT)


# [5단계] 출처카드 읽기: CSV 경로 또는 구글 시트 제목
def read_source_cards(source="출처카드"):
    """A CSV path, or (in Colab) the title of a Google Sheet."""
    if str(source).lower().endswith(".csv") or Path(str(source)).exists():
        return read_table(source)
    try:
        from google.colab import auth
    except ImportError:
        raise CollectError("구글 시트는 Colab에서만 읽을 수 있어요. 여기서는 CSV 경로를 주세요") from None
    import gspread
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    client = gspread.authorize(creds)
    try:
        sheet = client.open(source).sheet1
    except gspread.SpreadsheetNotFound:
        files = client.list_spreadsheet_files()
        title = pick_sheet_title(source, [f["name"] for f in files])
        print(f"'{title}'를 {source}로 읽었어요")
        sheet = client.open_by_key(next(f["id"] for f in files if f["name"] == title)).sheet1
    return pd.DataFrame(sheet.get_all_records())


# [5단계] 시트 이름이 정확히 같지 않을 때 띄어쓰기만 다른 시트 찾기
def pick_sheet_title(wanted, titles):
    """The one title equal to wanted once all whitespace is removed; stops on zero or several."""
    squeeze = lambda s: re.sub(r"\s+", "", str(s))
    matches = [t for t in titles if squeeze(t) == squeeze(wanted)]
    if not matches:
        raise CollectError(f"'{wanted}'라는 구글 시트를 못 찾았어요. 시트 이름을 확인해 주세요")
    if len(matches) > 1:
        raise CollectError(f"'{wanted}'와 띄어쓰기만 다른 시트가 {len(matches)}개예요: {matches}. "
                           f"쓸 시트 하나만 이름을 '{wanted}'로 바꿔 주세요")
    return matches[0]


# [3·5단계] 출처카드에 적힌 파일만 읽어서 공통 표로. 못 읽는 파일은 이유와 함께 건너뜀
def collect_from_cards(cards, files_dir):
    """Read files listed on the source cards. Only CSV/XLSX already shaped as 항목/값/단위/기준일.

    An optional 공시일 card column fills 공시일 the same way 기준일 does: a cell in the file wins.
    """
    missing = [c for c in CARD_COLUMNS if c not in cards.columns]
    if missing:
        raise CollectError(f"출처카드에 없는 칸: {missing}")
    results, skipped = [], []
    for card in cards.to_dict("records"):
        name = str(card["파일명"]).strip()
        path = Path(files_dir) / name
        if not path.exists():
            skipped.append((name, "files/ 폴더에 이 이름의 파일이 없어요"))
            continue
        if path.suffix.lower() not in (".csv", ".xlsx"):
            skipped.append((name, "PDF·글 자료는 4회차에 AI가 읽어요"))
            continue
        table = read_table(path)
        if not {"항목", "값"} <= set(table.columns):
            skipped.append((name, "'항목·값·단위·기준일' 칸 모양이 아니에요. 3단계 예시처럼 items로 바꿔 주세요. "
                                  f"지금 칸: {[str(col) for col in table.columns]}"))
            continue
        level_match = re.match(r"\s*([1-5])", str(card.get("수집 방법") or ""))
        disclosed = None if is_blank(card.get("공시일")) else card["공시일"]
        results.append(collect_table(
            table, level=int(level_match.group(1)) if level_match else 5, source=card["출처"], source_url=card["URL"],
            reliability=card["신뢰도"], base_date=card["기준일"], disclosed=disclosed, note=f"출처카드: {name}",
        ))
    for name, why in skipped:
        print(f"   ⏭ {name}: {why}")
    if not results:
        raise SkipStep(f"출처카드 {len(cards)}줄 중 읽을 수 있는 숫자 파일이 없어요")
    return pd.concat(results, ignore_index=True)


# 키 읽기: Colab이면 Secrets, 내 컴퓨터면 .env. 값은 절대 출력하지 않음
def get_key(name):
    """Colab Secrets first, then .env / environment. Never print the value."""
    try:
        from google.colab import userdata
    except ImportError:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            pass
        return os.getenv(name) or None
    try:
        return userdata.get(name) or None
    except Exception:
        return None

STEP_RESULTS.clear()
print("✅ 공통 도구 준비 완료")

✅ 공통 도구 준비 완료


In [4]:
#@title ⚙️ 엔진 도구 (고치지 말고 실행만) { display-mode: "form" }
"""KFA AI agent study: report engine skeleton.

Adds row ids and a numbered source list on top of the shared record table from
collect.py, computes metrics from those rows, and checks the numbers written in
claims against them. Engine columns go on copies only; the 15 shared columns never
change. Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 안내] 이 셀은 고치지 않고 실행만 해요.
    with_ids()        공통 표의 줄마다 이름표(rid)를 붙여요. 같은 사실이 두 줄이면 멈춰요
    build_sources()   출처 목록 [S1] [S2] ...을 만들고, 줄마다 어느 출처인지(source_id) 붙여요
    compute_metrics() 지표 정의 표(METRIC_SPECS)대로 지표 [M1] [M2] ...을 계산해요. 입력 줄이 없으면 값을 비우고 이유를 적어요
    check_claims()    주장에 적은 숫자를 지표나 원본 줄과 대조해서 PASS / FIX / REVIEW로 판정해요
    render_check()    주장 목록·지표 표·숫자 대조 표·출처 목록을 마크다운 파일 하나로 저장해요
    claims_prompt()   Gemini에게 보낼 질문(규칙 + 공통 표 + 지표 표 + 글)을 만들어요
    ask_claims()      Gemini 답을 원문 그대로 저장해요. 질문이 저장된 것과 똑같으면 다시 부르지 않아요
    use_backup()      Gemini가 안 될 때, 질문이 똑같으면 스터디장이 미리 받아 둔 답(backup/)을 써요
    load_claims()     저장된 답을 주장 목록으로 읽어요. 깨진 주장은 REVIEW로 넘겨요
"""

import json
import re
import time
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Literal
from urllib.parse import quote

import pandas as pd


RID_PARTS = ["종목코드", "항목", "기준", "기준일", "출처"]
SOURCE_KEY = ["출처", "출처 URL"]
SOURCE_COLUMNS = ["source_id", "출처", "출처 URL", "수집 방법", "신뢰도", "기준일 범위", "줄 수"]
METRIC_COLUMNS = ["metric_id", "지표", "값", "단위", "계산식", "입력 rid", "source_id", "기준일", "상태", "이유"]
CHECK_COLUMNS = ["claim_id", "문장", "근거 id", "적은 숫자", "적은 단위", "실제 값", "판정", "이유", "출처 번호"]
METRIC_BASIS = "연결"
COMPUTED = "계산됨"
PASS, FIX, CHECK = "PASS", "FIX", "REVIEW"
UNITS = {
    "원": ("원", Decimal(1)),
    "천원": ("원", Decimal(10) ** 3),
    "백만원": ("원", Decimal(10) ** 6),
    "억원": ("원", Decimal(10) ** 8),
    "십억원": ("원", Decimal(10) ** 9),
    "조원": ("원", Decimal(10) ** 12),
    "%": ("%", Decimal(1)),
}


@dataclass(frozen=True)
class MetricSpec:
    """growth: inputs = (item,), compared with the same item one year earlier.
    ratio: inputs = (numerator item, denominator item), same date."""
    name: str
    unit: str
    formula: str
    kind: Literal["growth", "ratio"]
    inputs: tuple


METRIC_SPECS = (
    MetricSpec("매출 성장률", "%", "(올해 매출액(연간) − 작년 매출액(연간)) ÷ 작년 매출액(연간) × 100",
               "growth", ("매출액(연간)",)),
    MetricSpec("영업이익률", "%", "영업이익(연간) ÷ 매출액(연간) × 100", "ratio", ("영업이익(연간)", "매출액(연간)")),
    MetricSpec("부채비율", "%", "부채총계 ÷ 자본총계 × 100", "ratio", ("부채총계", "자본총계")),
)


def _text(value):
    return "" if is_blank(value) else str(value)


def _source_order(sid):
    return int(sid[1:])


def _require_tagged(records):
    missing = [col for col in ("rid", "source_id") if col not in records.columns]
    if missing:
        raise CollectError(f"공통 표에 {missing} 칸이 없어요. build_sources(with_ids(표)) 결과를 넣어 주세요")


def with_ids(records):
    """Copy with rid = 종목코드|항목|기준|기준일|출처; a repeated rid is the same fact twice and stops."""
    out = records.copy()
    out["rid"] = ["|".join(_text(v) for v in row) for row in out[RID_PARTS].itertuples(index=False)]
    repeated = out.loc[out["rid"].duplicated(keep=False), "rid"].unique().tolist()
    if repeated:
        raise CollectError(
            f"같은 사실(종목코드|항목|기준|기준일|출처)이 두 줄 이상 있어요: {repeated[:5]}. "
            "같은 자료를 두 번 붙였는지 확인해 주세요")
    return out


def build_sources(records):
    """Return (sources, records copy with source_id).

    One source = one (출처, 출처 URL) pair, so each DART filing keeps its own URL.
    Numbers follow the sorted pairs, not row order, so reordering cells keeps every S#.
    """
    out = records.copy()
    keys = [tuple(_text(v) for v in row) for row in out[SOURCE_KEY].itertuples(index=False)]
    ids = {key: f"S{n}" for n, key in enumerate(sorted(set(keys)), 1)}
    out["source_id"] = [ids[key] for key in keys]
    rows = []
    for (name, url), sid in ids.items():
        part = out[out["source_id"] == sid]
        row = {"source_id": sid, "출처": name, "출처 URL": url, "줄 수": len(part)}
        for col in ("수집 방법", "신뢰도"):
            values = part[col].map(_text).unique().tolist()
            if len(values) > 1:
                raise CollectError(f"[{sid}] {name}: 한 출처인데 {col}이(가) 여러 개예요 {values}. 하나로 맞춰 주세요")
            row[col] = values[0]
        dates = sorted(d for d in part["기준일"].map(_text) if d)
        if dates:
            row["기준일 범위"] = dates[0] if dates[0] == dates[-1] else f"{dates[0]} ~ {dates[-1]}"
        else:
            row["기준일 범위"] = ""
        rows.append(row)
    return pd.DataFrame(rows, columns=SOURCE_COLUMNS), out


def _find_input(records, stock_code, item, base_date):
    """Return (row, None) for exactly one usable consolidated row, else (row or None, reason)."""
    label = f"{item} {base_date}"
    same = records[(records["종목코드"].map(_text) == stock_code) & (records["항목"].map(_text) == item)
                   & (records["기준일"].map(_text) == base_date)]
    rows = same[same["기준"].map(_text) == METRIC_BASIS]
    if rows.empty:
        hint = " (다른 기준 줄만 있음)" if not same.empty else ""
        return None, f"{label}: 연결 줄 없음{hint}"
    if len(rows) > 1:
        return None, f"{label}: 연결 줄이 {len(rows)}개라 하나를 고를 수 없음 ({rows['출처'].tolist()})"
    row = rows.iloc[0]
    if row["상태"] == REVIEW:
        return row, f"{label}: 입력 줄이 확인 필요"
    if is_blank(row["값"]):
        return row, f"{label}: 값 없음"
    return row, None


def compute_metrics(records, specs, stock_code, base_date):
    """One row per spec, M# in spec order. A metric whose inputs are missing, flagged,
    ambiguous or have a denominator <= 0 keeps an empty value with 확인 필요 and the reasons."""
    _require_tagged(records)
    prev_date = f"{int(base_date[:4]) - 1}{base_date[4:]}"
    out = []
    for n, spec in enumerate(specs, 1):
        if spec.kind == "growth":
            wanted = [(spec.inputs[0], base_date), (spec.inputs[0], prev_date)]
        else:
            wanted = [(item, base_date) for item in spec.inputs]
        found, reasons = [], []
        for item, date in wanted:
            row, reason = _find_input(records, stock_code, item, date)
            if row is not None:
                found.append(row)
            if reason:
                reasons.append(reason)
        value = None
        if not reasons:
            first, second = (float(row["값"]) for row in found)
            if second <= 0:
                label = f"{wanted[1][0]} {wanted[1][1]}"
                if spec.kind == "growth":
                    reasons.append(f"기준값 {label}이(가) 0 이하(적자): 흑자전환 / 적자지속 / 적자확대로 표기")
                else:
                    reasons.append(f"분모 {label}이(가) 0 이하라 비율이 의미 없음")
            else:
                value = ((first - second) if spec.kind == "growth" else first) / second * 100
        out.append({
            "metric_id": f"M{n}", "지표": spec.name, "값": value, "단위": spec.unit, "계산식": spec.formula,
            "입력 rid": [row["rid"] for row in found],
            "source_id": sorted({row["source_id"] for row in found}, key=_source_order),
            "기준일": base_date, "상태": REVIEW if reasons else COMPUTED, "이유": "; ".join(reasons),
        })
    metrics = pd.DataFrame(out, columns=METRIC_COLUMNS)
    metrics["값"] = pd.Series([row["값"] for row in out], dtype=object)
    return metrics


def _written_number(number):
    """Exact decimal of the number as written, so '46.80' keeps its two decimals."""
    if isinstance(number, bool):
        return None
    try:
        value = Decimal(str(number).strip().replace(",", ""))
    except InvalidOperation:
        return None
    return value if value.is_finite() else None


def _unit_scales(unit, actual_unit):
    """Return ((written scale, actual scale), None) or (None, reason).

    Identical unit text compares 1:1 whatever the unit is, because the rid or M# already
    fixes what is measured. Blank units are refused, or two blanks would match as equal text.
    """
    written, source = "".join(_text(unit).split()), "".join(_text(actual_unit).split())
    if not written or not source:
        return None, f"단위 없음 (문장 '{_text(unit)}' / 근거 '{_text(actual_unit)}')"
    if written == source:
        return (Decimal(1), Decimal(1)), None
    if written in UNITS and source in UNITS and UNITS[written][0] == UNITS[source][0]:
        return (UNITS[written][1], UNITS[source][1]), None
    return None, (f"단위가 안 맞음: 문장 '{_text(unit)}' / 근거 '{_text(actual_unit)}' "
                  "(단위 글자가 같거나, 원·천원·백만원·억원·십억원·조원끼리만 바꿔 비교)")


def _check_one(number, unit, actual, actual_unit):
    """Return (actual in the written unit or None, verdict, reason)."""
    written = _written_number(number)
    if written is None:
        return None, CHECK, f"적은 숫자 '{number}'를 숫자로 못 읽음"
    scales, problem = _unit_scales(unit, actual_unit)
    if problem:
        return None, CHECK, problem
    scale, actual_scale = scales
    actual_base = Decimal(str(actual)) * actual_scale
    tolerance = Decimal("0.5").scaleb(written.as_tuple().exponent) * scale
    shown = float(actual_base / scale)
    if abs(written * scale - actual_base) <= tolerance:
        return shown, PASS, ""
    return shown, FIX, f"실제 값은 {_format_actual(shown, written, unit)}"


def _format_actual(actual, written, unit):
    """Two more decimals than the written number, so a FIX shows how far off it is."""
    places = max(-written.as_tuple().exponent, 0) + 2
    return f"{actual:,.{places}f} {unit}"


def _lookup(evidence_id, metrics, records):
    """Return (value, unit, source ids, reason the value cannot be used or None)."""
    if not evidence_id:
        return None, None, [], "근거 id가 비어 있음 (표에 뒷받침하는 줄이 없음)"
    if evidence_id.startswith("M"):
        hit = metrics[metrics["metric_id"] == evidence_id]
        if hit.empty:
            return None, None, [], f"지표 {evidence_id}를 못 찾음"
        row = hit.iloc[0]
        if row["상태"] == REVIEW:
            return None, None, list(row["source_id"]), f"근거 지표가 확인 필요: {row['이유']}"
        return row["값"], row["단위"], list(row["source_id"]), None
    hit = records[records["rid"] == evidence_id]
    if hit.empty:
        return None, None, [], "근거 rid를 공통 표에서 못 찾음"
    row = hit.iloc[0]
    if row["상태"] == REVIEW:
        return None, None, [row["source_id"]], "근거 줄이 확인 필요"
    if is_blank(row["값"]):
        return None, None, [row["source_id"]], "근거 줄에 값 없음"
    return row["값"], _text(row["단위"]), [row["source_id"]], None


def check_claims(claims, metrics, records):
    """One row per (claim, evidence). Evidence ids starting with M are metrics, others are rids.

    claims: [{"claim_id": "C-1", "문장": "...", "근거": [(id, number, unit), ...]}].
    The number may be text ("46.80", "1,100") so trailing zeros keep their precision.
    A claim carrying "형식 오류" (set by load_claims) yields one REVIEW row and nothing else.
    출처 번호 collects the S# of every evidence row, or of a metric's input rows.
    """
    _require_tagged(records)
    out = []
    for claim in claims:
        evidence = claim.get("근거") or []
        rows, sources = [], set()
        problem = claim.get("형식 오류")
        if problem or not evidence:
            rows.append({"근거 id": "", "적은 숫자": None, "적은 단위": "", "실제 값": None, "판정": CHECK,
                         "이유": f"AI 답변 형식 오류: {problem}" if problem else "근거 없음"})
            evidence = []
        for evidence_id, number, unit in evidence:
            actual, actual_unit, sids, problem = _lookup(evidence_id, metrics, records)
            sources.update(sids)
            if problem:
                shown, verdict, reason = None, CHECK, problem
            else:
                shown, verdict, reason = _check_one(number, unit, actual, actual_unit)
            rows.append({"근거 id": evidence_id, "적은 숫자": number, "적은 단위": unit, "실제 값": shown,
                         "판정": verdict, "이유": reason})
        label = "".join(f"[{sid}]" for sid in sorted(sources, key=_source_order))
        for row in rows:
            out.append({"claim_id": claim.get("claim_id"), "문장": claim.get("문장"), **row, "출처 번호": label})
    return pd.DataFrame(out, columns=CHECK_COLUMNS)


def _cell(value):
    return _text(value).replace("|", "\\|").replace("\n", " ")


def _table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(_cell(v) for v in row) + " |" for row in rows]
    return lines


def _claim_number_rows(checked):
    for _, row in checked.iterrows():
        number, unit, actual = row["적은 숫자"], row["적은 단위"], row["실제 값"]
        written = _written_number(number)
        yield [
            f"{row['claim_id']} {row['문장']}",
            "" if is_blank(number) else f"{number} {unit}",
            "" if is_blank(actual) or written is None else _format_actual(actual, written, unit),
            f"{row['판정']}: {row['이유']}" if row["판정"] == CHECK else row["판정"],
        ]


def render_check(checked, metrics, sources, path):
    """Write claims, metrics, number checks and sources as one markdown file; return its path."""
    claims = checked.drop_duplicates("claim_id")
    lines = ["# 숫자 대조 결과", "", "## 주장", ""]
    lines += [f"- {row['claim_id']} {row['문장']} {row['출처 번호']}".rstrip() for _, row in claims.iterrows()]
    lines += ["", "## 지표", ""]
    lines += _table(["번호", "지표", "값", "계산식", "출처", "기준일"], [
        [row["metric_id"], row["지표"],
         f"확인 필요: {row['이유']}" if is_blank(row["값"]) else f"{row['값']:,.2f} {row['단위']}",
         row["계산식"], "".join(f"[{sid}]" for sid in row["source_id"]), row["기준일"]]
        for _, row in metrics.iterrows()])
    lines += ["", "## 숫자 대조", ""]
    lines += _table(["주장", "적은 숫자", "실제 값", "판정"], _claim_number_rows(checked))
    lines += ["", "## 출처", ""]
    lines += _table(["번호", "이름", "기준일 범위", "원본 URL", "신뢰도"], [
        [f"[{row['source_id']}]", row["출처"], row["기준일 범위"], row["출처 URL"], row["신뢰도"]]
        for _, row in sources.iterrows()])
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


BACKUP_URL = REPO_RAW_URL + "backup/"
RETRY_CODES = (429, 500, 503)
RETRY_WAITS = (5, 10, 20)
CLAIM_RULES = """아래 글에서 숫자가 들어간 주장을 모두 찾아, 각 숫자의 근거를 공통 표나 지표 표에서 골라 줘.

규칙
- 출력은 JSON 배열 하나만: [{"claim_id": "C-1", "문장": "...", "근거": [["근거 id", "숫자", "단위"]]}]
- claim_id는 C-1, C-2 ... 순서대로 붙여
- 문장은 글에 적힌 그대로 옮겨
- 근거 id는 공통 표의 rid나 지표 표의 metric_id(M1 ...)를 글자 그대로 복사해. 표에 없는 id를 만들지 마
- 숫자와 단위는 문장에 적힌 글자 그대로 적어 (예: "97.1", "조 원"). 단위를 바꾸거나 다시 계산하지 마
- 문장 속 숫자마다 근거를 하나씩 적어. 숫자가 가리키는 항목의 줄이나 지표를 골라. 값이 맞는지는 판단하지 마
- 증감, 포인트 차이 같은 변화량은 그 변화량 자체가 표에 있을 때만 고르고, 없으면 근거 id를 ""로 비워 (수준 지표에 변화량을 대지 마)
- 그 밖에는 항목 자체가 표에 없을 때만 근거 id를 ""로 비워
- 숫자가 없는 주장은 "근거": []로 적어"""


def claims_prompt(records, metrics, text):
    """Question text for Gemini: rules, the tables it may cite, then the text to read."""
    rows = records[["rid", "항목", "값", "단위", "기준일"]].to_csv(index=False, lineterminator="\n")
    metric_rows = metrics[["metric_id", "지표", "값", "단위"]].to_csv(index=False, lineterminator="\n")
    return f"{CLAIM_RULES}\n\n[공통 표]\n{rows}\n[지표 표]\n{metric_rows}\n[글]\n{text.strip()}\n"


def _call_gemini(prompt, api_key, model, fallback_model):
    """Return (answer text, model that answered). Same retry rule as setup/00_setup_check.ipynb:
    wait 5/10/20 s on 429/500/503, then try the fallback model once."""
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=api_key)
    config = types.GenerateContentConfig(response_mime_type="application/json", temperature=0)

    def ask(name):
        return client.models.generate_content(model=name, contents=prompt, config=config).text or ""

    try:
        for attempt, wait in enumerate((*RETRY_WAITS, None)):
            try:
                return ask(model), model
            except Exception as error:
                if getattr(error, "code", None) not in RETRY_CODES:
                    raise
                if wait is None:
                    break
                print(f"⏳ Gemini가 붐벼요. {wait}초 기다렸다가 다시 시도해요 ({attempt + 1}/{len(RETRY_WAITS)})")
                time.sleep(wait)
        print(f"⏳ 계속 안 돼서 예비 모델({fallback_model})로 한 번 더 시도해요")
        return ask(fallback_model), fallback_model
    except Exception as error:
        raise CollectError(_gemini_failure(getattr(error, "code", None), type(error).__name__)) from None


def _gemini_failure(code, error_name):
    """Member-facing reason a Gemini call failed. Only server-side trouble says 'try later';
    a key or model problem names what to fix instead."""
    if code in (400, 401, 403):
        return f"Gemini가 키를 받아 주지 않았어요 ({code}). Secrets의 GEMINI_API_KEY 값을 다시 확인해 주세요 (SETUP.md 9번)"
    if code == 404:
        return "Gemini 모델 이름을 찾을 수 없어요 (404). 스터디장에게 알려 주세요"
    if code in RETRY_CODES:
        return f"Gemini 서버가 계속 붐비거나 오류가 나요 ({code}). 내 설정 문제가 아니에요. 잠시 뒤 다시 실행해 주세요"
    return f"Gemini를 부르지 못했어요 ({code or error_name}). 인터넷 연결을 확인하고 잠시 뒤 다시 실행해 주세요"


def _record_path(path):
    return path.with_name(f"{path.stem}_기록.json")


def ask_claims(prompt, api_key, path, model, fallback_model, overwrite=False):
    """Save Gemini's raw answer at path and a record (question, model, time) beside it; return path.

    The saved answer is reused only when the saved question is identical to this one,
    so editing the text never checks an old answer. The record is removed before the
    answer is rewritten, so a record on disk always belongs to the answer next to it.
    """
    path = Path(path)
    record_path = _record_path(path)
    if not overwrite and path.exists() and record_path.exists():
        try:
            saved = json.loads(record_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            saved = {}
        if saved.get("질문") == prompt:
            print(f"♻ 같은 질문의 저장된 답을 다시 써요 ({saved.get('모델')}, {saved.get('호출 시각')})")
            return path
    if not api_key:
        raise SkipStep("Gemini 키가 없어요. Secrets에 GEMINI_API_KEY를 등록하고 노트북 액세스를 켠 뒤 다시 실행해 주세요 (SETUP.md 9번)")
    answer, used_model = _call_gemini(prompt, api_key, model, fallback_model)
    path.parent.mkdir(parents=True, exist_ok=True)
    record_path.unlink(missing_ok=True)
    path.write_text(answer, encoding="utf-8")
    record = {"질문": prompt, "모델": used_model, "호출 시각": now_kst()}
    record_path.write_text(json.dumps(record, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"✅ Gemini 답 저장: {path} ({used_model})")
    return path


def use_backup(prompt, path, name):
    """Fall back to the answer the study lead saved in backup/<name>.txt when Gemini fails.

    Used only when the backup's question is identical to this one, so a member who changed
    the text or the company never gets an answer to a different question. Returns path or None.
    """
    path = Path(path)
    try:
        record = json.loads(http_get(BACKUP_URL + quote(f"{name}_기록.json")).content.decode("utf-8"))
        if record.get("질문") != prompt:
            print("⏭ 미리 받아 둔 답은 기본 글·기본 종목용이라 지금 질문에는 쓸 수 없어요. 위 ⚠ 줄대로 고친 뒤 다시 실행해 주세요")
            return None
        answer = http_get(BACKUP_URL + quote(f"{name}.txt")).content.decode("utf-8")
    except (CollectError, json.JSONDecodeError):
        print("⏭ 미리 받아 둔 답이 없어요. 위 ⚠ 줄대로 고친 뒤 다시 실행해 주세요")
        return None
    record_path = _record_path(path)
    record_path.unlink(missing_ok=True)
    path.write_text(answer, encoding="utf-8")
    record_path.write_text(json.dumps(record, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"🗂 Gemini 호출이 안 돼서 스터디장이 미리 받아 둔 답을 썼어요 ({record.get('모델')}, {record.get('호출 시각')})")
    return path


def _salvage(text):
    """Split a broken answer into claim objects; a piece that does not parse stays a string.
    Claims hold no nested objects, so every '{' starts a new claim."""
    decoder, items = json.JSONDecoder(), []
    start = text.find("{")
    while start != -1:
        try:
            item, end = decoder.raw_decode(text, start)
        except json.JSONDecodeError:
            end = text.find("{", start + 1)
            items.append(text[start:end if end != -1 else len(text)])
            start = end
            continue
        items.append(item)
        start = text.find("{", end)
    return items


def _claim_from_answer(item, n, seen):
    if not isinstance(item, dict):
        found = re.search(r'"claim_id"\s*:\s*"([^"]+)"', item) if isinstance(item, str) else None
        claim_id = found.group(1) if found and found.group(1) not in seen else f"AI-{n}"
        seen.add(claim_id)
        return {"claim_id": claim_id, "문장": "", "근거": [],
                "형식 오류": f"JSON이 깨짐: {_text(item).strip()[:80]}"}
    problems = []
    claim_id = item.get("claim_id")
    if not isinstance(claim_id, str) or not claim_id.strip():
        problems.append("claim_id 없음")
        claim_id = f"AI-{n}"
    elif claim_id in seen:
        problems.append(f"claim_id {claim_id} 중복")
        claim_id = f"{claim_id} (AI-{n})"
    seen.add(claim_id)
    sentence = item.get("문장")
    if not isinstance(sentence, str):
        problems.append("문장 없음")
    evidence = item.get("근거")
    if not isinstance(evidence, list) or any(not isinstance(ev, list) or len(ev) != 3 for ev in evidence):
        problems.append("근거는 [근거 id, 숫자, 단위] 목록이어야 함")
        evidence = []
    claim = {"claim_id": claim_id, "문장": _text(sentence),
             "근거": [(_text(ev_id).strip(), number, _text(unit)) for ev_id, number, unit in evidence]}
    if problems:
        claim["형식 오류"] = "; ".join(problems)
    return claim


def load_claims(path):
    """Read a saved Gemini answer into check_claims input. Never raises on a bad answer:
    each claim that cannot be read becomes a claim carrying "형식 오류"."""
    text = Path(path).read_text(encoding="utf-8")
    body = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", text)
    try:
        items = json.loads(body)
        items = items if isinstance(items, list) else [items]
    except json.JSONDecodeError:
        items = _salvage(body)
    seen = set()
    claims = [_claim_from_answer(item, n, seen) for n, item in enumerate(items, 1)]
    return claims or [{"claim_id": "AI-1", "문장": "", "근거": [], "형식 오류": "AI 답변에서 주장을 하나도 못 찾음"}]

print("✅ 엔진 도구 준비 완료")

✅ 엔진 도구 준비 완료


In [5]:
DART_API_KEY = get_key("DART_API_KEY")
GEMINI_API_KEY = get_key("GEMINI_API_KEY")
print("DART 키:", "등록됨 ✅" if DART_API_KEY else "없음 ⏭ (Secrets에 DART_API_KEY + 노트북 액세스 켜기)")
print("Gemini 키:", "등록됨 ✅" if GEMINI_API_KEY else "없음 ⏭ (없으면 기본 글일 때만 미리 받아 둔 답을 써요)")

DART 키: 등록됨 ✅
Gemini 키: 등록됨 ✅


## 1. DART에서 두 해 받기

3회차와 같아요. 사업보고서의 **연결** 재무제표 6개 계정을 `YEAR`와 그 전년도로 받아요.

In [6]:
dart_years = []
for year in (YEAR - 1, YEAR):
    result = run_step(f"DART {year} 사업보고서", 1, lambda: collect_dart(STOCK_CODE, year, 4, "CFS", DART_API_KEY))
    if result is not None:
        dart_years.append(result)

dart = pd.concat(dart_years, ignore_index=True) if dart_years else None
if dart is not None:
    display(dart[["종목명", "항목", "값", "단위", "기준", "기준일", "공시일", "상태"]])

✅ DART 2024 사업보고서: 6줄 (확인 필요 0줄)
✅ DART 2025 사업보고서: 6줄 (확인 필요 0줄)


,종목명,항목,값,단위,기준,기준일,공시일,상태
0,SK하이닉스,자산총계,119855209000000,원,연결,2024-12-31,2025-03-19,형식 통과
1,SK하이닉스,자본총계,73915704000000,원,연결,2024-12-31,2025-03-19,형식 통과
2,SK하이닉스,부채총계,45939505000000,원,연결,2024-12-31,2025-03-19,형식 통과
3,SK하이닉스,영업이익(연간),23467319000000,원,연결,2024-12-31,2025-03-19,형식 통과
4,SK하이닉스,당기순이익(연간),19796902000000,원,연결,2024-12-31,2025-03-19,형식 통과
5,SK하이닉스,매출액(연간),66192960000000,원,연결,2024-12-31,2025-03-19,형식 통과
6,SK하이닉스,자산총계,176107659000000,원,연결,2025-12-31,2026-03-17,형식 통과
7,SK하이닉스,자본총계,120666751000000,원,연결,2025-12-31,2026-03-17,형식 통과
8,SK하이닉스,부채총계,55440908000000,원,연결,2025-12-31,2026-03-17,형식 통과
9,SK하이닉스,영업이익(연간),47206319000000,원,연결,2025-12-31,2026-03-17,형식 통과


## 1-1. 내 소스 붙이기 (선택)

3회차에 만든 `my_records(code)`를 아래 자리에 붙여 넣고 `MY_TARGETS`에 종목코드를 적으면, 내 소스 줄도 공통 표에 들어가서 Gemini가 근거로 고를 수 있어요.
비워 두면 ⏭로 넘어가고 공통 표는 DART만 그대로예요.

In [7]:
MY_LEVEL = 1
MY_TARGETS = []

def my_records(code):
    raise SkipStep("아직 내 소스를 안 채웠어요")

mine_parts = []
for code in MY_TARGETS:
    result = run_step(f"내 소스 {code}", MY_LEVEL, lambda: my_records(code))
    if result is not None:
        mine_parts.append(result)

if mine_parts:
    dart = pd.concat([f for f in (dart, *mine_parts) if f is not None], ignore_index=True)
    print(f"✅ 내 소스 {sum(len(f) for f in mine_parts)}줄을 공통 표에 붙였어요")
else:
    print("⏭ 내 소스 없음: 공통 표는 DART만으로 계속해요")

⏭ 내 소스 없음: 공통 표는 DART만으로 계속해요


## 2. 줄 이름표(rid)와 출처 번호([S#])

Gemini는 글 속 숫자마다 아래 표의 **rid**(줄 이름표) 하나를 골라 근거로 적어요. [S#]는 그 줄이 어느 공시에서 왔는지예요.

In [8]:
dart_sources, dart_tagged = None, None
if dart is None:
    print("⏭ DART 결과가 없어서 건너뛰어요 (위 셀의 ❌·⏭ 메모를 먼저 확인)")
else:
    try:
        dart_sources, dart_tagged = build_sources(with_ids(dart))
    except CollectError as error:
        print("❌", error)
    else:
        display(dart_sources)
        display(dart_tagged[["source_id", "rid", "항목", "값", "단위", "기준일"]])

,source_id,출처,출처 URL,수집 방법,신뢰도,기준일 범위,줄 수
0,S1,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2024-12-31,6
1,S2,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2025-12-31,6


,source_id,rid,항목,값,단위,기준일
0,S1,000660|자산총계|연결|2024-12-31|DART 단일회사 전체 재무제표 API,자산총계,119855209000000,원,2024-12-31
1,S1,000660|자본총계|연결|2024-12-31|DART 단일회사 전체 재무제표 API,자본총계,73915704000000,원,2024-12-31
2,S1,000660|부채총계|연결|2024-12-31|DART 단일회사 전체 재무제표 API,부채총계,45939505000000,원,2024-12-31
3,S1,000660|영업이익(연간)|연결|2024-12-31|DART 단일회사 전체 재무제...,영업이익(연간),23467319000000,원,2024-12-31
4,S1,000660|당기순이익(연간)|연결|2024-12-31|DART 단일회사 전체 재무...,당기순이익(연간),19796902000000,원,2024-12-31
5,S1,000660|매출액(연간)|연결|2024-12-31|DART 단일회사 전체 재무제표...,매출액(연간),66192960000000,원,2024-12-31
6,S2,000660|자산총계|연결|2025-12-31|DART 단일회사 전체 재무제표 API,자산총계,176107659000000,원,2025-12-31
7,S2,000660|자본총계|연결|2025-12-31|DART 단일회사 전체 재무제표 API,자본총계,120666751000000,원,2025-12-31
8,S2,000660|부채총계|연결|2025-12-31|DART 단일회사 전체 재무제표 API,부채총계,55440908000000,원,2025-12-31
9,S2,000660|영업이익(연간)|연결|2025-12-31|DART 단일회사 전체 재무제...,영업이익(연간),47206319000000,원,2025-12-31


## 3. 지표 [M#]

매출 성장률·영업이익률·부채비율을 위 줄들로만 계산해요. 계산에 쓴 줄(입력 rid)과 출처 번호가 같이 나와요. 입력 줄이 없으면 값을 비우고 이유를 적어요.

In [9]:
metrics = None
if dart_tagged is None:
    print("⏭ 줄 이름표가 없어서 건너뛰어요")
else:
    metrics = compute_metrics(dart_tagged, METRIC_SPECS, STOCK_CODE, f"{YEAR}-{QUARTER_END[4]}")
    display(metrics)

,metric_id,지표,값,단위,계산식,입력 rid,source_id,기준일,상태,이유
0,M1,매출 성장률,46.762851,%,(올해 매출액(연간) − 작년 매출액(연간)) ÷ 작년 매출액(연간) × 100,[000660|매출액(연간)|연결|2025-12-31|DART 단일회사 전체 재무제...,"[S1, S2]",2025-12-31,계산됨,
1,M2,영업이익률,48.59283,%,영업이익(연간) ÷ 매출액(연간) × 100,[000660|영업이익(연간)|연결|2025-12-31|DART 단일회사 전체 재무...,[S2],2025-12-31,계산됨,
2,M3,부채비율,45.945472,%,부채총계 ÷ 자본총계 × 100,[000660|부채총계|연결|2025-12-31|DART 단일회사 전체 재무제표 A...,[S2],2025-12-31,계산됨,


## 4. Gemini에게 글을 보내 주장 저장

질문(규칙 + 위 두 표 + 글)을 보내고, 답을 **받은 그대로** 파일에 저장해요. 같은 질문으로 다시 돌리면 Gemini를 또 부르지 않고 저장한 답을 써요(♻).
Gemini가 안 되면, 기본 글·기본 종목일 때만 스터디장이 미리 받아 둔 답을 써요(🗂).

In [10]:
GEMINI_MODEL = "gemini-3.8-flash"
GEMINI_FALLBACK_MODEL = "gemini-3.5-flash-lite"
AI_FILE = f"w04_ai_claims_{STOCK_CODE}_{YEAR}.txt"

claims_path = None
if metrics is None:
    print("⏭ 지표가 없어서 건너뛰어요")
else:
    prompt = claims_prompt(dart_tagged, metrics, TEXT)
    try:
        claims_path = ask_claims(prompt, GEMINI_API_KEY, AI_FILE, GEMINI_MODEL, GEMINI_FALLBACK_MODEL)
    except (CollectError, SkipStep) as error:
        print("⚠", error)
        claims_path = use_backup(prompt, AI_FILE, f"ai_claims_{STOCK_CODE}_{YEAR}")

✅ Gemini 답 저장: w04_ai_claims_000660_2025.txt (gemini-3.8-flash)


## 5. 저장한 답 읽기

먼저 Gemini 답을 **원문 그대로** 보여주고, 그걸 주장 목록으로 읽어요. 형식이 깨진 주장은 멈추지 않고 그 주장만 REVIEW로 넘겨요.

In [11]:
claims = None
if claims_path is None:
    print("⏭ 저장한 답이 없어서 건너뛰어요")
else:
    print(Path(claims_path).read_text(encoding="utf-8"))
    claims = load_claims(claims_path)
    display(pd.DataFrame(claims))

[
  {
    "claim_id": "C-1",
    "문장": "SK하이닉스의 2025년 매출은 97.1조 원으로 전년보다 46.8% 늘었다.",
    "근거": [
      [
        "000660|매출액(연간)|연결|2025-12-31|DART 단일회사 전체 재무제표 API",
        "97.1",
        "조 원"
      ],
      [
        "M1",
        "46.8",
        "%"
      ]
    ]
  },
  {
    "claim_id": "C-2",
    "문장": "영업이익률은 52%를 기록했고, 전년 대비 13.1%포인트 높아졌다.",
    "근거": [
      [
        "M2",
        "52",
        "%"
      ],
      [
        "",
        "13.1",
        "%포인트"
      ]
    ]
  },
  {
    "claim_id": "C-3",
    "문장": "부채비율은 46% 수준이다.",
    "근거": [
      [
        "M3",
        "46",
        "%"
      ]
    ]
  },
  {
    "claim_id": "C-4",
    "문장": "부채비율은 전년보다 16.2%p 낮아졌다.",
    "근거": [
      [
        "",
        "16.2",
        "%p"
      ]
    ]
  },
  {
    "claim_id": "C-5",
    "문장": "HBM 수요가 실적을 이끌었다.",
    "근거": []
  }
]


,claim_id,문장,근거
0,C-1,SK하이닉스의 2025년 매출은 97.1조 원으로 전년보다 46.8% 늘었다.,[(000660|매출액(연간)|연결|2025-12-31|DART 단일회사 전체 재무...
1,C-2,"영업이익률은 52%를 기록했고, 전년 대비 13.1%포인트 높아졌다.","[(M2, 52, %), (, 13.1, %포인트)]"
2,C-3,부채비율은 46% 수준이다.,"[(M3, 46, %)]"
3,C-4,부채비율은 전년보다 16.2%p 낮아졌다.,"[(, 16.2, %p)]"
4,C-5,HBM 수요가 실적을 이끌었다.,[]


## 6. 숫자 대조

주장에 적힌 숫자를 근거(지표나 원본 줄)와 같은 단위로 맞춰 비교해요. 적은 숫자의 마지막 자리 반올림 차이까지는 PASS예요(46.8%면 ±0.05%p).

In [12]:
checked = None
if claims is None:
    print("⏭ 주장 목록이 없어서 건너뛰어요")
else:
    checked = check_claims(claims, metrics, dart_tagged)
    display(checked)

,claim_id,문장,근거 id,적은 숫자,적은 단위,실제 값,판정,이유,출처 번호
0,C-1,SK하이닉스의 2025년 매출은 97.1조 원으로 전년보다 46.8% 늘었다.,000660|매출액(연간)|연결|2025-12-31|DART 단일회사 전체 재무제표...,97.1,조 원,97.146675,PASS,,[S1][S2]
1,C-1,SK하이닉스의 2025년 매출은 97.1조 원으로 전년보다 46.8% 늘었다.,M1,46.8,%,46.762851,PASS,,[S1][S2]
2,C-2,"영업이익률은 52%를 기록했고, 전년 대비 13.1%포인트 높아졌다.",M2,52,%,48.592830,FIX,실제 값은 48.59 %,[S2]
3,C-2,"영업이익률은 52%를 기록했고, 전년 대비 13.1%포인트 높아졌다.",,13.1,%포인트,NaN,REVIEW,근거 id가 비어 있음 (표에 뒷받침하는 줄이 없음),[S2]
4,C-3,부채비율은 46% 수준이다.,M3,46,%,45.945472,PASS,,[S2]
5,C-4,부채비율은 전년보다 16.2%p 낮아졌다.,,16.2,%p,NaN,REVIEW,근거 id가 비어 있음 (표에 뒷받침하는 줄이 없음),
6,C-5,HBM 수요가 실적을 이끌었다.,,None,,NaN,REVIEW,근거 없음,


## 7. 결과 파일 미리보기

주장 목록([S#] 포함), 지표 표, 숫자 대조 표, 출처 목록을 마크다운 파일 하나로 저장하고 화면에 그대로 보여줘요.

In [13]:
from IPython.display import Markdown

if checked is None:
    print("⏭ 대조 결과가 없어서 건너뛰어요")
else:
    report = render_check(checked, metrics, dart_sources, f"w04_check_{STOCK_CODE}_{YEAR}.md")
    print("💾 저장:", report)
    display(Markdown(report.read_text(encoding="utf-8")))

💾 저장: w04_check_000660_2025.md


# 숫자 대조 결과

## 주장

- C-1 SK하이닉스의 2025년 매출은 97.1조 원으로 전년보다 46.8% 늘었다. [S1][S2]
- C-2 영업이익률은 52%를 기록했고, 전년 대비 13.1%포인트 높아졌다. [S2]
- C-3 부채비율은 46% 수준이다. [S2]
- C-4 부채비율은 전년보다 16.2%p 낮아졌다.
- C-5 HBM 수요가 실적을 이끌었다.

## 지표

| 번호 | 지표 | 값 | 계산식 | 출처 | 기준일 |
|---|---|---|---|---|---|
| M1 | 매출 성장률 | 46.76 % | (올해 매출액(연간) − 작년 매출액(연간)) ÷ 작년 매출액(연간) × 100 | [S1][S2] | 2025-12-31 |
| M2 | 영업이익률 | 48.59 % | 영업이익(연간) ÷ 매출액(연간) × 100 | [S2] | 2025-12-31 |
| M3 | 부채비율 | 45.95 % | 부채총계 ÷ 자본총계 × 100 | [S2] | 2025-12-31 |

## 숫자 대조

| 주장 | 적은 숫자 | 실제 값 | 판정 |
|---|---|---|---|
| C-1 SK하이닉스의 2025년 매출은 97.1조 원으로 전년보다 46.8% 늘었다. | 97.1 조 원 | 97.147 조 원 | PASS |
| C-1 SK하이닉스의 2025년 매출은 97.1조 원으로 전년보다 46.8% 늘었다. | 46.8 % | 46.763 % | PASS |
| C-2 영업이익률은 52%를 기록했고, 전년 대비 13.1%포인트 높아졌다. | 52 % | 48.59 % | FIX |
| C-2 영업이익률은 52%를 기록했고, 전년 대비 13.1%포인트 높아졌다. | 13.1 %포인트 |  | REVIEW: 근거 id가 비어 있음 (표에 뒷받침하는 줄이 없음) |
| C-3 부채비율은 46% 수준이다. | 46 % | 45.95 % | PASS |
| C-4 부채비율은 전년보다 16.2%p 낮아졌다. | 16.2 %p |  | REVIEW: 근거 id가 비어 있음 (표에 뒷받침하는 줄이 없음) |
| C-5 HBM 수요가 실적을 이끌었다. |  |  | REVIEW: 근거 없음 |

## 출처

| 번호 | 이름 | 기준일 범위 | 원본 URL | 신뢰도 |
|---|---|---|---|---|
| [S1] | DART 단일회사 전체 재무제표 API | 2024-12-31 | https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20250319000665 | 상 |
| [S2] | DART 단일회사 전체 재무제표 API | 2025-12-31 | https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20260317000635 | 상 |


> 💾 답 원문·기록·결과 파일은 Colab 런타임이 끝나면 사라져요. **파일 → GitHub에 사본 저장**으로 `w04_검증기.ipynb`를 내 레포에 올리면 답 원문과 대조 결과까지 같이 올라가요.

4번 셀 문구: ✅ Gemini 답 저장: w04_ai_claims_000660_2025.txt (gemini-3.8-flash)
숫자 대조 결과: PASS 3 · FIX 1 · REVIEW 3